In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from src.chess_features import (split_by_puzzle, topk, heuristic_score, fit_eval,
                                ORIGINAL_NUM, BASE_COLS, MATE_COLS, DYNAMIC_COLS,
                                NEW_COLS, CAT_COLS)

Path("results").mkdir(exist_ok=True)
cand = pd.read_csv("data/candidates.csv", dtype={"PuzzleId": str})
train, test = split_by_puzzle(cand)

print("Train puzzles:", train["PuzzleId"].nunique(), "| test puzzles:", test["PuzzleId"].nunique())
print("Puzzles in both:", len(set(train["PuzzleId"]) & set(test["PuzzleId"])))   # must be 0

Train puzzles: 4000 | test puzzles: 1000
Puzzles in both: 0


In [2]:
results = {}
results["Random"] = topk(test, np.zeros(len(test)))
results["Checks/captures heuristic"] = topk(test, heuristic_score(test).values)

res_logit, logit = fit_eval(train, test, ORIGINAL_NUM, CAT_COLS)
results["Logistic regression"] = res_logit

main_results = pd.DataFrame(results).T
main_results.to_csv("results/main_results.csv")

names = logit.named_steps["pre"].get_feature_names_out()
coefs = pd.Series(logit.named_steps["clf"].coef_[0], index=names).sort_values()
coefs.to_csv("results/coefficients.csv", header=["coefficient"])

main_results

,top1,top3
Random,0.051,0.142
Checks/captures heuristic,0.360,0.727
Logistic regression,0.541,0.820


In [3]:
dynamics = BASE_COLS + MATE_COLS + DYNAMIC_COLS

ablation = {
    "Heuristic features only": fit_eval(train, test, BASE_COLS, [])[0],
    "+ mate": fit_eval(train, test, BASE_COLS + MATE_COLS, [])[0],
    "+ opp_replies, attacked": fit_eval(train, test, dynamics, [])[0],
    "+ piece type": fit_eval(train, test, dynamics, CAT_COLS)[0],
    "All original features": fit_eval(train, test, ORIGINAL_NUM, CAT_COLS)[0],
    "+ new tactical features": fit_eval(train, test, ORIGINAL_NUM + NEW_COLS, CAT_COLS)[0],
}
ablation_df = pd.DataFrame(ablation).T
ablation_df.to_csv("results/ablation.csv")
ablation_df

,top1,top3
Heuristic features only,0.403,0.741
+ mate,0.463,0.759
"+ opp_replies, attacked",0.557,0.807
+ piece type,0.550,0.824
All original features,0.541,0.820
+ new tactical features,0.618,0.846


In [4]:
features = ORIGINAL_NUM + NEW_COLS

res_lr, _ = fit_eval(train, test, features, CAT_COLS, kind="logit")
res_mlp, mlp = fit_eval(train, test, features, CAT_COLS, kind="mlp")

nn_df = pd.DataFrame({"Logistic regression": res_lr, "Neural network (32,16)": res_mlp}).T
nn_df.to_csv("results/neural_network.csv")
nn_df

,top1,top3
Logistic regression,0.618,0.846
"Neural network (32,16)",0.661,0.872


In [5]:
rows = []
for seed in range(5):
    tr, te = split_by_puzzle(cand, seed=seed)
    rows.append({"seed": seed, "model": "Random", **topk(te, np.zeros(len(te)))})
    rows.append({"seed": seed, "model": "Heuristic", **topk(te, heuristic_score(te).values)})
    for name, kind in [("Logistic regression", "logit"), ("Neural network", "mlp")]:
        res, _ = fit_eval(tr, te, features, CAT_COLS, kind=kind)
        rows.append({"seed": seed, "model": name, **res})

repeated = pd.DataFrame(rows)
repeated.to_csv("results/repeated_splits.csv", index=False)
repeated.groupby("model")[["top1", "top3"]].agg(["mean", "std"]).round(3)

top1          top3       
                      mean    std   mean    std
model                                          
Heuristic            0.385  0.013  0.723  0.011
Logistic regression  0.609  0.009  0.839  0.015
Neural network       0.673  0.014  0.870  0.014
Random               0.051  0.006  0.148  0.016

In [6]:
wide = repeated.pivot(index="seed", columns="model", values=["top1", "top3"])
diff = pd.DataFrame({
    "top1": wide["top1"]["Neural network"] - wide["top1"]["Logistic regression"],
    "top3": wide["top3"]["Neural network"] - wide["top3"]["Logistic regression"],
})
print(diff.round(3))
print(diff.agg(["mean", "std"]).round(3))

       top1   top3
seed              
0     0.055  0.022
1     0.077  0.032
2     0.057  0.031
3     0.070  0.035
4     0.059  0.036
       top1   top3
mean  0.064  0.031
std   0.009  0.006
